In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv("(KULFFI)_16S_Genus_level.csv")
bac_cols = [c for c in df.columns if c.startswith('d__Bacteria')]

df_shannon = pd.read_csv("(KULFFI)_Shannon.csv")
meta = df_shannon[['sample-id', 'Feature', 'Cluster']].rename(columns={'sample-id': 'index', 'Cluster': 'Donor_meta'})
df = pd.merge(df, meta, on='index', how='inner')

df['Feature'] = df['Feature_x'].combine_first(df['Feature_y'])
df.loc[df['Feature'] == 'FEC', 'Feature'] = 'Feces'
df.loc[df['Feature'] == 'CUL', 'Feature'] = 'Control'

df_sub = df[df['Feature'].isin(['Feces', 'Control'])].copy()
total_abund = df_sub[bac_cols].sum(axis=1)

global_phylum_data = {}
for col in bac_cols:
    phylum = 'Other'
    for p in col.split(';'):
        p = p.strip()
        if p.startswith('p__'):
            phylum = p.replace('p__', '')
            break
    if phylum not in global_phylum_data:
        global_phylum_data[phylum] = pd.Series(0.0, index=df_sub.index)
    global_phylum_data[phylum] += df_sub[col]

df_global_phy = pd.DataFrame(global_phylum_data).div(total_abund, axis=0) * 100
df_global_phy['Condition'] = df_sub['Feature']
global_phylum_feces_mean = df_global_phy.groupby('Condition').mean().loc['Feces']

phylum_order = global_phylum_feces_mean.sort_values(ascending=False).index.tolist()
if 'Other' in phylum_order:
    phylum_order.remove('Other')

distinct_colors = [
    '#1f77b4', '#d62728', '#2ca02c', '#ff7f0e', '#9467bd',
    '#8c564b', '#e377c2', '#bcbd22', '#17becf', '#393b79',
    '#8c6d31', '#843c39', '#7b4173', '#5254a3', '#637939',
    '#ce6dbd', '#fd8d3c', '#f16913', '#d94801', '#807dba'
]

base_phylum_colors = {}
for i, p in enumerate(phylum_order):
    base_phylum_colors[p] = distinct_colors[i % len(distinct_colors)]

def get_taxa_plot_data(level_prefix, top_n):
    taxa_data = {}
    taxon_to_phylum = {}

    for col in bac_cols:
        parts = col.split(';')
        phylum = 'Other'
        taxon = ''

        for p in parts:
            p = p.strip()
            if p.startswith('p__'):
                phylum = p.replace('p__', '')
            if p.startswith(level_prefix):
                taxon = p.replace(level_prefix, '')

        if not taxon:
            continue

        if taxon not in taxa_data:
            taxa_data[taxon] = pd.Series(0.0, index=df_sub.index)
        taxa_data[taxon] += df_sub[col]
        taxon_to_phylum[taxon] = phylum

    df_taxa = pd.DataFrame(taxa_data).div(total_abund, axis=0) * 100
    df_taxa['Condition'] = df_sub['Feature']

    mean_abund = df_taxa.groupby('Condition').mean()
    feces_mean = mean_abund.loc['Feces'].sort_values(ascending=False)

    top_taxa = feces_mean.index[:top_n].tolist()

    def sort_key(t):
        p = taxon_to_phylum.get(t, 'Other')
        return (global_phylum_feces_mean.get(p, 0), feces_mean[t])

    sorted_taxa = sorted(top_taxa, key=sort_key, reverse=True)

    color_map = {}
    seen_phyla = []
    for t in sorted_taxa:
        p = taxon_to_phylum.get(t, 'Other')
        if p not in seen_phyla:
            seen_phyla.append(p)

    for p in seen_phyla:
        group = [t for t in sorted_taxa if taxon_to_phylum.get(t, 'Other') == p]
        n = len(group)
        base_c = base_phylum_colors.get(p, '#333333')

        if n == 1:
            shades = [base_c]
        else:
            shades = sns.light_palette(base_c, n_colors=n+3, reverse=True).as_hex()[:n]

        for t, hex_color in zip(group, shades):
            color_map[t] = hex_color

    mean_abund['Other_combined'] = mean_abund.drop(columns=top_taxa, errors='ignore').sum(axis=1)
    ordered_cols = sorted_taxa + ['Other_combined']
    plot_df = mean_abund[ordered_cols].rename(columns={'Other_combined': 'Other'})
    plot_df = plot_df.loc[['Feces', 'Control']]
    plot_colors = [color_map[t] for t in sorted_taxa] + ['#A9A9A9']

    return plot_df, plot_colors

plt.rcParams['font.weight'] = 'normal'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

levels_config = [
    ('p__', 'Phylum', 10),
    ('c__', 'Class', 14),
    ('o__', 'Order', 18),
    ('f__', 'Family', 22)
]

fig, axes = plt.subplots(2, 2, figsize=(18, 14), dpi=600)
axes = axes.flatten()

for i, (prefix, title, top_n) in enumerate(levels_config):
    ax = axes[i]
    plot_df, plot_colors = get_taxa_plot_data(prefix, top_n)

    bottom = np.zeros(2)
    for j, taxon in enumerate(plot_df.columns):
        ax.bar(plot_df.index, plot_df[taxon], bottom=bottom, label=taxon,
               color=plot_colors[j], edgecolor='black', linewidth=1.0)
        bottom += plot_df[taxon]

    ax.set_title(title, fontsize=20, fontweight='bold', pad=15)
    if i % 2 == 0:
        ax.set_ylabel("Relative Abundance (%)", fontsize=18, fontweight='bold')

    ax.tick_params(axis='y', labelsize=16, width=1.5, length=6)
    ax.tick_params(axis='x', width=1.5, length=6)
    ax.set_xticklabels(plot_df.index, fontsize=18, fontweight='bold')

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=12, frameon=False)

    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

plt.tight_layout()

output_file = "Supplementary_Figure_S2b.pdf"
plt.savefig(output_file, dpi=600, format='pdf', bbox_inches='tight', transparent=True)